# Homework 5, Problem 2: P-wave Zoeppritz Coefficients at the Core-Mantle Boundary

**Solid-fluid interface** — Medium 1 (lower mantle) is solid; Medium 2 (outer core) is fluid.
The fluid cannot sustain shear stress, so there is no transmitted S-wave ($T_{ps} = 0$).
This reduces the Zoeppritz system to $3 \times 3$ with unknowns $[R_{pp}, R_{ps}, T_{pp}]^T$.

In [ ]:
# Cell 1: Import dependencies and define physical parameters
# This notebook solves the 3x3 Zoeppritz system for P-wave incidence
# at the Core-Mantle Boundary (solid-fluid interface) — Homework 5, Problem 2

import numpy as np
from numpy.lib import scimath  # scimath.arcsin handles complex args beyond critical angles
import matplotlib.pyplot as plt
import os

# --- CMB physical parameters ---
# Medium 1: Lower Mantle (solid)
density_1 = 5.5e3       # rho_1 [kg/m^3]
p_velocity_1 = 13.7e3   # alpha_1 [m/s] — P-wave velocity
s_velocity_1 = 7.2e3    # beta_1  [m/s] — S-wave velocity

# Medium 2: Outer Core (fluid)
density_2 = 9.9e3       # rho_2 [kg/m^3]
p_velocity_2 = 8.0e3    # alpha_2 [m/s] — P-wave velocity
s_velocity_2 = 0.0      # beta_2  [m/s] — fluid cannot sustain shear

print('CMB parameters loaded.')
print(f'Medium 1 (Lower Mantle, solid): rho={density_1:.1e}, vp={p_velocity_1:.1e}, vs={s_velocity_1:.1e}')
print(f'Medium 2 (Outer Core, fluid):  rho={density_2:.1e}, vp={p_velocity_2:.1e}, vs={s_velocity_2:.1e}')

In [ ]:
# Cell 2: Core 3x3 Zoeppritz solver for P-wave incidence at solid-fluid interface

def calculate_cmb_zoeppritz(incident_angle_deg, density_1, p_velocity_1,
                            s_velocity_1, density_2, p_velocity_2):
    """
    Compute reflection and transmission coefficients for an incident P-wave
    at the Core-Mantle Boundary (solid-fluid interface) using the
    3x3 Zoeppritz system.
    
    The fluid (medium 2) cannot sustain shear stress, so T_ps = 0.
    Unknowns: x = [R_pp, R_ps, T_pp]^T
    
    Parameters
    ----------
    incident_angle_deg : float
        Incident P-wave angle [degrees]
    density_1, p_velocity_1, s_velocity_1 : float
        Density [kg/m^3], P-wave and S-wave velocities [m/s] in medium 1 (solid)
    density_2, p_velocity_2 : float
        Density [kg/m^3], P-wave velocity [m/s] in medium 2 (fluid)
    
    Returns
    -------
    dict : {
        'Rpp': reflected P-wave coefficient (complex),
        'Rps': reflected S-wave coefficient (complex),
        'Tpp': transmitted P-wave coefficient (complex),
        'Tps': 0.0 (no transmitted S-wave in fluid),
        'theta_S1': reflected S angle [rad],
        'theta_P2': transmitted P angle [rad],
        'ray_parameter': ray parameter p
    }
    """
    # Convert incidence angle to radians
    i_rad = np.deg2rad(incident_angle_deg)
    
    # === Step 1: Snell's Law ===
    # Ray parameter p = sin(i) / alpha_1 (constant across all waves)
    alpha1 = p_velocity_1
    beta1  = s_velocity_1
    alpha2 = p_velocity_2
    rho1 = density_1
    rho2 = density_2
    
    p = np.sin(i_rad) / alpha1
    
    # === Step 2: Compute other wave angles via Snell's law ===
    # Use scimath.arcsin to handle complex angles beyond critical incidence
    theta_P1 = i_rad                                    # incident P angle
    theta_S1 = scimath.arcsin(p * beta1)                # reflected S angle
    theta_P2 = scimath.arcsin(p * alpha2)               # transmitted P angle
    
    # === Step 3: Build the 3x3 Zoeppritz matrix equation M * x = b ===
    #
    # Boundary conditions at a solid-fluid interface:
    #   Row 0 — normal displacement continuity (w)
    #   Row 1 — normal stress continuity (sigma_zz)
    #   Row 2 — tangential stress = 0 (sigma_xz, fluid condition)
    #
    # Unknown vector: x = [R_pp, R_ps, T_pp]^T   (T_ps = 0)
    
    # Short names for readability
    sin_i  = np.sin(theta_P1)
    cos_i  = np.cos(theta_P1)
    sin_j  = np.sin(theta_S1)
    cos_j  = np.cos(theta_S1)
    sin_f  = np.sin(theta_P2)
    cos_f  = np.cos(theta_P2)
    
    cos_2j = np.cos(2.0 * theta_S1)  # cos(2*theta_S1)
    sin_2j = np.sin(2.0 * theta_S1)  # sin(2*theta_S1)
    sin_2i = np.sin(2.0 * theta_P1)  # sin(2*theta_P1)
    
    # --- Build the 3x3 coefficient matrix M ---
    M = np.array([
        [cos_i,                               sin_j,                               cos_f],
        [-rho1 * alpha1 * cos_2j,             rho1 * beta1 * sin_2j,               rho2 * alpha2],
        [(beta1 / alpha1) * sin_2i,           -cos_2j,                             0.0]
    ])
    
    # --- Build the right-hand side vector b ---
    # Represents the incident P-wave contributions
    b = np.array([
        cos_i,
        rho1 * alpha1 * cos_2j,
        (beta1 / alpha1) * sin_2i
    ])
    
    # === Step 4: Solve the linear system ===
    x = np.linalg.solve(M, b)
    
    # Return results as a dictionary for clarity
    return {
        'Rpp': x[0],           # Reflected P-wave coefficient
        'Rps': x[1],           # Reflected S-wave coefficient (mode-converted)
        'Tpp': x[2],           # Transmitted P-wave coefficient
        'Tps': 0.0 + 0.0j,     # No transmitted S-wave in fluid
        'theta_S1': theta_S1,
        'theta_P2': theta_P2,
        'ray_parameter': p
    }

# Quick test: normal incidence (i=0)
result = calculate_cmb_zoeppritz(0, density_1, p_velocity_1, s_velocity_1,
                                 density_2, p_velocity_2)
# Expected: Rpp ≈ (rho2*alpha2 - rho1*alpha1) / (rho1*alpha1 + rho2*alpha2)
#         = (79.2e6 - 75.35e6) / (75.35e6 + 79.2e6) ≈ 0.0249
Z1 = density_1 * p_velocity_1
Z2 = density_2 * p_velocity_2
Rpp_expected = (Z2 - Z1) / (Z1 + Z2)
print(f'Verification at normal incidence (i=0):')
print(f'  Rpp = {result["Rpp"].real:.6f}  (expected: {Rpp_expected:.6f})')
print(f'  Rps = {result["Rps"].real:.6f}  (expected: 0)')
print(f'  Tpp = {result["Tpp"].real:.6f}  (expected: {1 - Rpp_expected:.6f})')
print(f'  Tps = {result["Tps"].real:.6f}  (expected: 0)')
print(f'  P-impedance: Z1={Z1:.2e}, Z2={Z2:.2e}')
print('Function ready.')

In [ ]:
# Cell 3: Batch computation across all incident angles and plot results

# --- Batch computation ---
angle_array = np.arange(0, 91, 0.5)  # 0 to 90 degrees, 0.5 degree steps
n_angles = len(angle_array)

# Pre-allocate arrays for coefficients (will be complex)
Rpp_arr = np.zeros(n_angles, dtype=complex)
Rps_arr = np.zeros(n_angles, dtype=complex)
Tpp_arr = np.zeros(n_angles, dtype=complex)

# Compute coefficients for each incident angle
for idx, angle in enumerate(angle_array):
    result = calculate_cmb_zoeppritz(angle, density_1, p_velocity_1,
                                     s_velocity_1, density_2, p_velocity_2)
    Rpp_arr[idx] = result['Rpp']
    Rps_arr[idx] = result['Rps']
    Tpp_arr[idx] = result['Tpp']

# --- Plotting ---
# Plot the absolute value (amplitude) of each coefficient.

# Compute y-axis limits dynamically from data
all_vals = np.concatenate([np.abs(Rpp_arr), np.abs(Rps_arr), np.abs(Tpp_arr)])
y_max = np.max(all_vals)
y_lim = y_max * 1.05  # 5% headroom

plt.figure(figsize=(10, 7))
plt.plot(angle_array, np.abs(Rpp_arr), 'b-',  linewidth=2, label=r'$|R_{pp}|$ (Reflected P)')
plt.plot(angle_array, np.abs(Rps_arr), 'r-',  linewidth=2, label=r'$|R_{ps}|$ (Reflected S)')
plt.plot(angle_array, np.abs(Tpp_arr), 'g--', linewidth=2, label=r'$|T_{pp}|$ (Transmitted P)')

# Mark critical angle for reflected S-wave: maximum reflected S angle
# occurs when sin(i) = 1, giving sin(theta_S1_max) = beta1 / alpha1
crit_S1 = np.rad2deg(np.arcsin(s_velocity_1 / p_velocity_1))
plt.axvline(crit_S1, color='cyan', linestyle=':', alpha=0.6)
plt.text(crit_S1 + 0.5, y_lim * 0.9, r'$i_{c}^{S1}$', color='cyan', fontsize=9)

# Note: alpha1 > alpha2 (13.7 > 8.0 km/s), so sin(theta_P2) < 1 for all i
#       and there is no transmitted P critical angle.

plt.xlabel('Incident Angle (degrees)', fontsize=12)
plt.ylabel('Displacement Coefficient Amplitude', fontsize=12)
plt.title('P-wave Zoeppritz Coefficients at Core-Mantle Boundary', fontsize=14)
plt.legend(fontsize=10, loc='best')
plt.grid(True, alpha=0.3)
plt.xlim(0, 90)
plt.ylim(0, y_lim)

# Ensure figures directory exists
os.makedirs('figures', exist_ok=True)

# Save figure to file
plt.savefig('figures/HW5_Q2_CMB_Zoeppritz.png', dpi=150, bbox_inches='tight')
print(f'Figure saved: figures/HW5_Q2_CMB_Zoeppritz.png')
print(f'Y-axis range: 0 to {y_lim:.3f}')
print(f'Critical angle (maximum reflected S-wave angle): {crit_S1:.2f} deg')
print(f'Note: alpha1 > alpha2 ({p_velocity_1} > {p_velocity_2} m/s) — no transmitted P critical angle')

# Handle headless display
import os as _os
if _os.environ.get('DISPLAY') or _os.name == 'nt':
    plt.show()
else:
    plt.close()